# Pipeline 1 — sequence OCR and entities

Select **Godfather (Python 3.11 | Torch + Paddle GPU)** and run top to bottom.

**Sequence → Magi boxes/associations → selected OCR → cross-page entities → text–entity records.**

Text-box order is immutable. `texts[i]`, `ocr[i]`, and association indices always refer to the same original box. Empty text stays in place. Entity IDs are scoped to the sequence (`E1`, `E2`, …); they are not GT speaker names.

Every sequence has its own cache folder. Stage files contain `data`, a signature, and a checksum; the manifest records inputs, settings, and stage metadata. This notebook does not edit the reference notebooks.

## 0. Imports and sequence settings

<details>
<summary>How this stage works</summary>

Torch imports first to avoid the Windows DLL conflict. Both models run directly in this kernel.

Change `SEQUENCE_ID`, `SPLIT`, or settings below, then rerun from this section. `FORCE` recomputes individual stages; reset flags afterwards. The entity threshold is **0.50**, matching the identity reference notebook's executed call.

Labels are loaded as separate supervision. They never enter inference or determine entity IDs.

</details>

In [1]:
import torch  # Must precede Paddle on this Windows installation.
import paddle
import gc
import copy
import hashlib
import json
import sys
import unicodedata
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from transformers import AutoModel
from paddleocr import PaddleOCR

print('Python:', sys.executable)
print('Torch:', torch.__version__, '| Paddle:', paddle.__version__)


c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
W1006 10:11:13.215000 34824 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Python: c:\Users\ESHAAN\BYTE\godfather.venv\Scripts\python.exe
Torch: 2.14.0+cu126 | Paddle: 3.2.0


In [2]:
MAGI_ROOT = Path(r'C:\Users\ESHAAN\BYTE\Applied-Task\Magi')
MODEL_PATH = Path(r'C:\MODELS\magiv2')
SPLIT = 'development'
SEQUENCE_ID = 'seq_0dc2e3d874c804fc'
LANGUAGE = 'en'
PREFERRED_OCR = 'paddle'
ENTITY_THRESHOLD = 0.50
CHARACTER_CLEANUP_SETTINGS = dict(min_iou=0.90, min_cosine=0.95,
                                  algorithm='contained_direct_outer_match_v1')
# Bump after replacing model weights without changing their file metadata.
MODEL_REVISION = 'local-magiv2-1'
FORCE = dict(magi_raw=False, embeddings=False, paddle_ocr=False,
             ocr_selection=False, character_cleanup=False, entities=False, text_entities=False)
# Bump the affected version when editing a stage's algorithm.
STAGE_VERSIONS = dict(magi_raw=1, embeddings=1, paddle_ocr=1,
                     ocr_selection=1, character_cleanup=1, entities=1, text_entities=1)
PADDLE_SETTINGS = dict(lang=LANGUAGE, device='gpu:0', ocr_version='PP-OCRv5',
                      use_doc_orientation_classify=False, use_doc_unwarping=False,
                      use_textline_orientation=False)
SELECTOR_SETTINGS = dict(prefer=PREFERRED_OCR, check='unicode_text_validity_v1')


In [3]:
SEQUENCE_ROOT = MAGI_ROOT / 'TaskData' / SPLIT / 'images' / SEQUENCE_ID
LABELS_PATH = MAGI_ROOT / 'TaskData' / SPLIT / 'labels.jsonl'
CACHE_DIR = MAGI_ROOT / 'paddleOCR-Magi' / 'cache' / SEQUENCE_ID
assert SEQUENCE_ROOT.is_dir(), f'Missing sequence: {SEQUENCE_ROOT}'
assert MODEL_PATH.is_dir(), f'Missing model: {MODEL_PATH}'
PAGE_FILES = sorted(SEQUENCE_ROOT.glob('*.png'))
assert PAGE_FILES, 'No PNG pages found'
chapter_pages = []
for path in PAGE_FILES:
    with Image.open(path) as image:
        chapter_pages.append(np.array(image.convert('L').convert('RGB')))

sequence_labels = None
with LABELS_PATH.open(encoding='utf-8') as file:
    for line in file:
        if line.strip():
            entry = json.loads(line)
            if entry['sequence_id'] == SEQUENCE_ID:
                sequence_labels = entry
                break
assert sequence_labels is not None, f'No labels for {SEQUENCE_ID}'
assert len(sequence_labels['pages']) == len(PAGE_FILES), 'Label/page count mismatch'
print('Sequence:', SEQUENCE_ID, '| pages:', [p.name for p in PAGE_FILES])
print('Cache:', CACHE_DIR)


Sequence: seq_0dc2e3d874c804fc | pages: ['01.png', '02.png', '03.png']
Cache: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache\seq_0dc2e3d874c804fc


## 1. Cache helpers

<details>
<summary>How this stage works</summary>

Files: `manifest.json`, `magi_raw.json`, `embeddings.json`, `paddle_ocr.json`, `ocr_selection.json`, `entities.json`, `text_entities.json`.

Pages use SHA-256 image hashes. Model Python/config files are hashed; large weights use size and modification time plus `MODEL_REVISION`. Package versions and settings enter the relevant signatures.

Dependencies: Magi → Paddle/embeddings; Magi + Paddle → selector; embeddings + Magi → entities; selector + entities + Magi → final mapping. Changing selector rules does not rerun GPU stages. Changing the clustering threshold reuses embeddings. Label changes update supervision metadata without invalidating inference.

Writes replace temporary files atomically. Use one writer per sequence; different sequences can run in parallel. Bump an affected `STAGE_VERSIONS` entry after editing its algorithm.

</details>

In [4]:
def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


def digest(data):
    text = json.dumps(json_ready(data), sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def file_hash(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(data), ensure_ascii=True,
                                    indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def model_fingerprint():
    files = []
    for path in sorted(MODEL_PATH.rglob('*')):
        if path.is_file() and path.suffix in {'.json', '.py', '.safetensors', '.bin', '.pt', '.pth'}:
            stat = path.stat()
            files.append(dict(name=str(path.relative_to(MODEL_PATH)), size=stat.st_size,
                              mtime_ns=stat.st_mtime_ns,
                              sha256=file_hash(path) if path.suffix in {'.json', '.py'} else None))
    return dict(path=str(MODEL_PATH), revision=MODEL_REVISION, files=files)


PACKAGES = {name: version(name) for name in ['torch', 'torchvision', 'transformers',
            'paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy', 'pillow']}
PAGE_METADATA = [dict(name=p.name, sha256=file_hash(p), width=image.shape[1], height=image.shape[0])
                 for p, image in zip(PAGE_FILES, chapter_pages)]
INPUTS = dict(sequence_id=SEQUENCE_ID, split=SPLIT, pages=PAGE_METADATA,
              preprocessing='grayscale_then_RGB')
MODEL_INFO = model_fingerprint()
manifest = dict(schema_version=1, inputs=INPUTS, model=MODEL_INFO, packages=PACKAGES,
                labels=dict(path=str(LABELS_PATH), sequence_sha256=digest(sequence_labels)), stages={})
MANIFEST_PATH = CACHE_DIR / 'manifest.json'
if MANIFEST_PATH.exists():
    try:
        old_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        if old_manifest.get('inputs') == INPUTS:
            manifest['stages'] = old_manifest.get('stages', {})
    except (ValueError, OSError):
        pass


def stage_ref(name):
    entry = manifest['stages'][name]
    return dict(signature=entry['signature'], data_hash=entry['data_hash'])


def cached_stage(name, settings, compute):
    signature = digest(dict(inputs=INPUTS, algorithm_version=STAGE_VERSIONS[name], settings=settings))
    path = CACHE_DIR / f'{name}.json'
    payload = None
    if path.exists() and not FORCE[name]:
        try:
            candidate = json.loads(path.read_text(encoding='utf-8'))
            if candidate['signature'] == signature and digest(candidate['data']) == candidate['data_hash']:
                payload = candidate
        except (ValueError, KeyError, OSError):
            pass
    if payload is None:
        print(f'RUN   {name}')
        data = json_ready(compute())
        payload = dict(signature=signature, data_hash=digest(data),
                       created_at=datetime.now(timezone.utc).isoformat(), data=data)
        write_json(path, payload)
    else:
        print(f'CACHE {name}')
    manifest['stages'][name] = dict(file=path.name, signature=signature,
                                  data_hash=payload['data_hash'], settings=settings,
                                  algorithm_version=STAGE_VERSIONS[name], created_at=payload['created_at'])
    write_json(MANIFEST_PATH, manifest)
    return payload['data']


## 2. Magi predictions and character embeddings

<details>
<summary>How this stage works</summary>

Uses FAFO's `do_chapter_wide_prediction(..., do_ocr=True)` flow. `magi_raw.json` retains original results.

Embeddings are calculated and L2-normalized while Magi is loaded, then cached for later clustering. Magi is released before Paddle to reduce GPU memory use. Cache hits do not load the model.

</details>

In [5]:
magi_model = None


def get_magi():
    global magi_model
    if magi_model is None:
        assert torch.cuda.is_available(), 'Torch CUDA unavailable'
        magi_model = AutoModel.from_pretrained(str(MODEL_PATH), trust_remote_code=True,
                                              local_files_only=True).cuda().eval()
    return magi_model


def predict_magi():
    with torch.inference_mode():
        return get_magi().do_chapter_wide_prediction(
            chapter_pages, {'images': [], 'names': []}, use_tqdm=True, do_ocr=True)


def predict_embeddings():
    boxes = [result['characters'] for result in magi_raw_results]
    if not any(boxes):
        return [[] for _ in boxes]
    with torch.inference_mode():
        embeddings = get_magi().predict_crop_embeddings(chapter_pages, boxes)
        return [torch.nn.functional.normalize(x, p=2, dim=1) if len(x) else x for x in embeddings]


def validate_magi(results):
    assert len(results) == len(chapter_pages), 'Magi page count mismatch'
    for result in results:
        assert len(result['texts']) == len(result['ocr']), 'Text/OCR count mismatch'
        assert len(result['characters']) == len(result['character_cluster_labels']), 'Character/cluster count mismatch'
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']), 'Invalid text association index'
            assert 0 <= char_index < len(result['characters']), 'Invalid character association index'


In [6]:
try:
    magi_raw_results = cached_stage('magi_raw',
        dict(model=MODEL_INFO, torch=PACKAGES['torch'], torchvision=PACKAGES['torchvision'],
             transformers=PACKAGES['transformers'], do_ocr=True), predict_magi)
    validate_magi(magi_raw_results)
    embeddings_per_page = cached_stage('embeddings',
        dict(magi=stage_ref('magi_raw'), model=MODEL_INFO, normalization='L2',
             torch=PACKAGES['torch'], transformers=PACKAGES['transformers']), predict_embeddings)
finally:
    magi_model = None
    gc.collect()
    torch.cuda.empty_cache()

assert len(embeddings_per_page) == len(magi_raw_results)
for result, embeddings in zip(magi_raw_results, embeddings_per_page):
    assert len(embeddings) == len(result['characters']), 'Character/embedding count mismatch'
    assert np.isfinite(np.asarray(embeddings)).all(), 'Nonfinite embeddings'
per_page_results = copy.deepcopy(magi_raw_results)
print('Text counts:', [len(r['texts']) for r in per_page_results])


RUN   magi_raw


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\torch\nn\modules\module.py:2593: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=Tr

RUN   embeddings
Text counts: [7, 7, 14]


## 3. Paddle OCR on original text boxes

<details>
<summary>How this stage works</summary>

Runs Paddle directly on GPU. Records retain page order and original text-box order. Coordinates are rounded outward and clipped only for cropping; original boxes are retained.

Invalid crops and per-box inference errors keep placeholder records, so selection can fall back to Magi. Set `FORCE['paddle_ocr'] = True` to retry cached errors.

</details>

In [7]:
def crop_box(image, box):
    values = np.asarray(box, dtype=float)
    if values.shape != (4,) or not np.isfinite(values).all():
        return None, None
    height, width = image.shape[:2]
    x1, y1, x2, y2 = values
    clipped = [int(np.clip(np.floor(x1), 0, width)), int(np.clip(np.floor(y1), 0, height)),
               int(np.clip(np.ceil(x2), 0, width)), int(np.clip(np.ceil(y2), 0, height))]
    x1, y1, x2, y2 = clipped
    if x2 <= x1 or y2 <= y1:
        return None, clipped
    return np.ascontiguousarray(image[y1:y2, x1:x2]), clipped


def predict_paddle():
    output = [[] for _ in chapter_pages]
    if not any(result['texts'] for result in magi_raw_results):
        return output
    assert paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0, 'Paddle CUDA unavailable'
    engine = PaddleOCR(**PADDLE_SETTINGS)
    try:
        for page_index, (image, result) in enumerate(zip(chapter_pages, magi_raw_results)):
            for text_index, box in enumerate(result['texts']):
                crop, clipped = crop_box(image, box)
                record = dict(page_index=page_index, text_index=text_index, text_id=f'T{text_index+1}',
                              box=box, crop_box=clipped, text='', lines=[], status='invalid_crop', error=None)
                if crop is not None:
                    try:
                        for prediction in engine.predict(input=crop):
                            texts, scores = prediction['rec_texts'], prediction['rec_scores']
                            assert len(texts) == len(scores), 'Paddle text/score count mismatch'
                            for text, score in zip(texts, scores):
                                record['lines'].append(dict(text=str(text), confidence=float(score)))
                        record['text'] = ' '.join(line['text'] for line in record['lines'])
                        record['status'] = 'text' if record['text'].strip() else 'empty'
                    except Exception as error:
                        record.update(text='', lines=[], status='error', error=str(error))
                        print(f'PG{page_index+1}:T{text_index+1} OCR error: {error}')
                output[page_index].append(record)
    finally:
        del engine
        gc.collect()
        paddle.device.cuda.empty_cache()
    return output


In [8]:
paddle_ocr_results = cached_stage('paddle_ocr',
    dict(magi=stage_ref('magi_raw'), paddle=PADDLE_SETTINGS,
         packages={k: PACKAGES[k] for k in ['paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy', 'pillow']}),
    predict_paddle)
print('Paddle records:', sum(len(page) for page in paddle_ocr_results))
print('Cached OCR errors:', sum(r['status'] == 'error' for page in paddle_ocr_results for r in page))


Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ESHAAN\.paddlex\official_models\PP-OCRv5_server_det`.


RUN   paddle_ocr


Creating model: ('en_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ESHAAN\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


Paddle records: 28
Cached OCR errors: 0


## 4. Select OCR without changing text order

<details>
<summary>How this stage works</summary>

Basic sanity means **valid text**, not correct recognition. No word-specific rules or spelling checks.

Select the only passing output; prefer Paddle when both pass. If only one is nonempty but fails validity (e.g. `...`), retain it and flag review. If both are empty or both nonempty outputs fail, select empty text and flag review. Case-insensitive normalized differences are marked `disagreement`.

Cache both raw OCRs, Paddle confidence/status/errors, and every decision. Selected text becomes `per_page_results[page]['ocr']`; original Magi OCR remains in `magi_ocr`.

</details>

In [9]:
def text_validity(text):
    if text is None:
        return dict(passes=False, reason='empty', text='')
    if not isinstance(text, str):
        return dict(passes=False, reason='not_a_string', text='')
    normalized = ' '.join(text.split())
    if not normalized:
        reason = 'empty'
    elif '\ufffd' in text or any(unicodedata.category(c) in {'Cc', 'Cs'} and c not in '\n\r\t' for c in text):
        reason = 'invalid_characters'
    elif not any(c.isalnum() for c in normalized):
        reason = 'no_letters_or_digits'
    else:
        reason = 'valid_text'
    return dict(passes=reason == 'valid_text', reason=reason, text=normalized)


def select_ocr(paddle_text, magi_text, prefer='paddle'):
    assert prefer in {'paddle', 'magi'}, 'Choose paddle or magi'
    checks = {'paddle': text_validity(paddle_text), 'magi': text_validity(magi_text)}
    passing = [source for source, check in checks.items() if check['passes']]
    nonempty = [source for source, check in checks.items() if check['text']]
    if len(passing) == 1:
        source, reason = passing[0], 'only_valid_output'
    elif len(passing) == 2:
        source, reason = prefer, 'both_valid_preferred'
    elif len(nonempty) == 1:
        source, reason = nonempty[0], 'only_nonempty_needs_review'
    else:
        source = None
        reason = 'both_empty' if all(c['reason'] == 'empty' for c in checks.values()) else 'both_invalid'
    return dict(paddle_text=paddle_text, magi_text=magi_text, validity=checks,
                selected_text=checks[source]['text'] if source else '', selected_source=source,
                reason=reason, needs_review=not bool(passing),
                disagreement=len(passing) == 2 and checks['paddle']['text'].casefold() != checks['magi']['text'].casefold())


def select_sequence_ocr():
    assert len(paddle_ocr_results) == len(magi_raw_results), 'Paddle page count mismatch'
    selected = []
    for page_index, (magi, paddle_records) in enumerate(zip(magi_raw_results, paddle_ocr_results)):
        assert len(paddle_records) == len(magi['texts']) == len(magi['ocr']), 'OCR count mismatch'
        decisions = []
        for text_index, (box, record, magi_text) in enumerate(zip(magi['texts'], paddle_records, magi['ocr'])):
            assert record['page_index'] == page_index and record['text_index'] == text_index, 'OCR order mismatch'
            assert record['box'] == box, 'OCR crop mismatch'
            decision = select_ocr(record['text'], magi_text, PREFERRED_OCR)
            decision.update(page_index=page_index, text_index=text_index, text_id=f'T{text_index+1}', box=box,
                            paddle_lines=record['lines'], paddle_status=record['status'], paddle_error=record['error'])
            decisions.append(decision)
        selected.append(decisions)
    return selected


In [10]:
ocr_selection = cached_stage('ocr_selection',
    dict(magi=stage_ref('magi_raw'), paddle=stage_ref('paddle_ocr'), selector=SELECTOR_SETTINGS),
    select_sequence_ocr)
per_page_results = copy.deepcopy(magi_raw_results)
assert len(ocr_selection) == len(per_page_results), 'Selected OCR page count mismatch'
for page_index, (result, decisions) in enumerate(zip(per_page_results, ocr_selection)):
    assert len(decisions) == len(result['texts']), 'Selected OCR count mismatch'
    for text_index, decision in enumerate(decisions):
        assert decision['page_index'] == page_index and decision['text_index'] == text_index
        assert decision['box'] == result['texts'][text_index], 'Selected OCR box mismatch'
    result['magi_ocr'] = list(result['ocr'])
    result['ocr'] = [record['selected_text'] for record in decisions]
for raw, selected in zip(magi_raw_results, per_page_results):
    assert raw['texts'] == selected['texts'] and len(raw['texts']) == len(selected['ocr'])
display(pd.DataFrame([dict(page=r['page_index']+1, text_id=r['text_id'],
                          paddle=r['paddle_text'], magi=r['magi_text'], selected=r['selected_text'],
                          source=r['selected_source'], reason=r['reason'], review=r['needs_review'],
                          disagreement=r['disagreement']) for page in ocr_selection for r in page]))


RUN   ocr_selection


,page,text_id,paddle,magi,selected,source,reason,review,disagreement
0,1,T1,THIS IS OUR OWN THING WE GOT HERE! IT'S GONNA ...,This is our OOW thing we got here! It's gonna ...,THIS IS OUR OWN THING WE GOT HERE! IT'S GONNA ...,paddle,both_valid_preferred,False,True
1,1,T2,"WHEN WE DO BUSINESS WITH SOMEONE LIKE DINO, IT...","When we do business with someone like Dino, it...","WHEN WE DO BUSINESS WITH SOMEONE LIKE DINO, IT...",paddle,both_valid_preferred,False,True
2,1,T3,HE CAN'T BE TRUSTED EXCEPT TO TRY AND PUT A CO...,HE CAN'T BE TRUSTED -- EXCEPT TO TRY AND PUT A...,HE CAN'T BE TRUSTED EXCEPT TO TRY AND PUT A CO...,paddle,both_valid_preferred,False,True
3,1,T4,BUT IF YOU WAN TO ROLL OVER FOR HIM...,but if you WANT to roll over for him...,BUT IF YOU WAN TO ROLL OVER FOR HIM...,paddle,both_valid_preferred,False,True
4,1,T5,...AT LEAST SHOW ME ENOUGH RESPECT NOT TO BEG ...,...at least show me enough respect not to beg ...,...AT LEAST SHOW ME ENOUGH RESPECT NOT TO BEG ...,paddle,both_valid_preferred,False,False
5,1,T6,"ISN'T THAT RIGHT, ARTHUR?","Isn't that right, Arthur?","ISN'T THAT RIGHT, ARTHUR?",paddle,both_valid_preferred,False,False
6,1,T7,e0A,...,e0A,paddle,only_valid_output,False,False
7,2,T1,"HEY, YOU'RE THE BOSS.","Hey, you're the boss.","HEY, YOU'RE THE BOSS.",paddle,both_valid_preferred,False,False
8,2,T2,I'VE GOT NO COMPLAINTS,I've got no complaints.,I'VE GOT NO COMPLAINTS,paddle,both_valid_preferred,False,True
9,2,T3,THAT KEEPS THINGS NICE AND SIMPLE FOR ME,THAT KEEPS THINGS NICE AND SIMPLE FOR ME.,THAT KEEPS THINGS NICE AND SIMPLE FOR ME,paddle,both_valid_preferred,False,True


## 4b. Remove duplicate character appearances

<details>
<summary>Containment plus embedding similarity, with redirected links</summary>

Keep the outer character box only when it fully contains the inner box, IoU is at least 0.90, and embedding cosine similarity is at least 0.95. These are adjustable heuristic thresholds, not a guarantee of identity. Mere containment does not remove a character.

Compare each inner box directly with an already retained outer box; do not merge through a chain of removed boxes. Surviving characters retain their original relative order. Keep the outer box's embedding, cluster label, and name. Redirect text-character links, deduplicate identical links, and remap character indices. Text boxes, OCR, text-tail links, and reading order stay unchanged.

`character_cleanup.json` stores cleaned pages, aligned embeddings, and the original-to-cleaned index map with IoU/cosine values for every removal. Raw Magi results and raw embeddings stay unchanged. Downstream entities use cleaned character indices; consult the audit to map them back to raw detections. No GT labels are used.

</details>


In [11]:
def remove_duplicate_characters(results, embeddings, min_iou=0.90, min_cosine=0.95):
    """Remove contained, visually similar duplicates; preserve all text indices."""
    assert 0 <= min_iou <= 1 and -1 <= min_cosine <= 1
    assert len(results) == len(embeddings)
    cleaned_pages, cleaned_embeddings, audits = [], [], []
    for page_index, (raw, vectors) in enumerate(zip(results, embeddings)):
        boxes = np.asarray(raw['characters'], dtype=float).reshape(-1, 4)
        assert len(vectors) == len(boxes), 'Character/embedding count mismatch'
        assert len(raw['character_cluster_labels']) == len(boxes)
        areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
        # Largest first. Equal areas prefer the earlier original character index.
        retained, representatives, decisions = [], {}, []
        for inner in sorted(range(len(boxes)), key=lambda i: (-areas[i], i)):
            representative = inner
            for outer in retained:
                a, b = boxes[inner], boxes[outer]
                if not np.isfinite(a).all() or not np.isfinite(b).all() or areas[inner] <= 0 or areas[outer] <= 0:
                    continue
                contained = a[0] >= b[0] and a[1] >= b[1] and a[2] <= b[2] and a[3] <= b[3]
                iou = float(areas[inner] / areas[outer]) if contained else 0.0
                if not contained or iou < min_iou:
                    continue
                x, y = np.asarray(vectors[inner], dtype=float), np.asarray(vectors[outer], dtype=float)
                assert x.ndim == y.ndim == 1 and x.shape == y.shape
                denominator = np.linalg.norm(x) * np.linalg.norm(y)
                if not np.isfinite(x).all() or not np.isfinite(y).all() or not np.isfinite(denominator) or denominator <= 0:
                    continue
                cosine = float(np.clip(np.dot(x, y) / denominator, -1, 1))
                if cosine >= min_cosine:
                    representative = outer
                    decisions.append(dict(removed_original_index=inner, retained_original_index=outer,
                                          smaller_box_coverage=1.0, iou=iou, cosine=cosine))
                    break
            representatives[inner] = representative
            if representative == inner:
                retained.append(inner)
        # Keep the original relative order of surviving character appearances.
        kept = sorted(retained)
        new_index = {old: new for new, old in enumerate(kept)}
        old_to_new = [new_index[representatives[i]] for i in range(len(boxes))]
        page = copy.deepcopy(raw)
        for key in ['characters', 'character_cluster_labels', 'character_names']:
            if key in raw:
                assert len(raw[key]) == len(boxes), f'Misaligned {key}'
                page[key] = [raw[key][i] for i in kept]
        links, seen = [], set()
        for text_index, char_index in raw['text_character_associations']:
            assert 0 <= text_index < len(raw['texts']) and 0 <= char_index < len(boxes)
            link = (text_index, old_to_new[char_index])
            if link not in seen:
                seen.add(link)
                links.append(list(link))
        page['text_character_associations'] = links
        assert page['texts'] == raw['texts'] and page['ocr'] == raw['ocr']
        cleaned_pages.append(page)
        cleaned_embeddings.append([vectors[i] for i in kept])
        audits.append(dict(page_index=page_index, kept_original_indices=kept,
                           original_to_cleaned_index=old_to_new, removals=decisions))
    return dict(pages=cleaned_pages, embeddings=cleaned_embeddings, audit=audits)


character_cleanup = cached_stage('character_cleanup',
    dict(magi=stage_ref('magi_raw'), embeddings=stage_ref('embeddings'),
         cleanup=CHARACTER_CLEANUP_SETTINGS),
    lambda: remove_duplicate_characters(magi_raw_results, embeddings_per_page,
        CHARACTER_CLEANUP_SETTINGS['min_iou'], CHARACTER_CLEANUP_SETTINGS['min_cosine']))
character_results = character_cleanup['pages']
character_embeddings_per_page = character_cleanup['embeddings']
validate_magi(character_results)
# Apply only character fields; preserve the selected OCR already in memory.
for result, cleaned in zip(per_page_results, character_results):
    for key in ['characters', 'character_cluster_labels', 'character_names', 'text_character_associations']:
        if key in cleaned:
            result[key] = copy.deepcopy(cleaned[key])
print('Character counts before:', [len(r['characters']) for r in magi_raw_results])
print('Character counts after: ', [len(r['characters']) for r in character_results])
display(pd.DataFrame([dict(page_index=page['page_index'], **decision)
                      for page in character_cleanup['audit'] for decision in page['removals']]))


RUN   character_cleanup
Character counts before: [11, 12, 8]
Character counts after:  [10, 12, 8]


,page_index,removed_original_index,retained_original_index,smaller_box_coverage,iou,cosine
0,0,4,5,1.0,0.937536,0.994321


## 5. Cross-page character entities

<details>
<summary>How this stage works</summary>

Adapted from `magiv2_sequence_identity_full.ipynb`, not the annotated viewer's alternate mutual-match algorithm.

Matching unit: Magi page-local character cluster (`PG1:CL1`). Score: maximum member-to-member cosine against an existing entity. Candidates are processed strongest-first. Each local cluster matches at most one entity, and each entity accepts at most one local cluster per page. Unmatched clusters create an `E` ID. Matching candidates can be sorted; text boxes never are.

Cache retains members, character→entity mapping, accepted scores, and strongest evidence pairs. Negative/unassigned cluster labels are treated as separate appearances.

</details>

In [12]:
def page_clusters(result, page_index):
    groups = {}
    for char_index, label in enumerate(result['character_cluster_labels']):
        cluster_id = f'PG{page_index+1}:CL{int(label)+1}' if int(label) >= 0 else f'PG{page_index+1}:C{char_index+1}:unassigned'
        groups.setdefault(cluster_id, []).append(char_index)
    return groups


def clusterize(results, embeddings, threshold=0.50):
    assert -1 <= threshold <= 1, 'Cosine threshold must be between -1 and 1'
    vectors = [np.asarray(page, dtype=float) for page in embeddings]
    assert len(results) == len(vectors)
    for result, page_vectors in zip(results, vectors):
        assert len(result['characters']) == len(page_vectors), 'Embedding count mismatch'
    entities, decisions, char_to_entity = {}, [], {}
    for page_index, result in enumerate(results):
        clusters = page_clusters(result, page_index)
        candidates = []
        for cluster_id, indices in clusters.items():
            for entity_id, entity in entities.items():
                best_score, best_pair = -float('inf'), None
                for current in indices:
                    for member in entity['members']:
                        score = float(np.dot(vectors[page_index][current], vectors[member['page_index']][member['character_index']]))
                        if score > best_score:
                            best_score = score
                            best_pair = dict(current=f'PG{page_index+1}:C{current+1}', existing=member['character_id'])
                if best_score >= threshold:
                    candidates.append(dict(cluster_id=cluster_id, entity_id=entity_id, score=best_score, best_pair=best_pair))
        candidates.sort(key=lambda candidate: candidate['score'], reverse=True)
        accepted, used_entities = {}, set()
        for candidate in candidates:
            if candidate['cluster_id'] not in accepted and candidate['entity_id'] not in used_entities:
                accepted[candidate['cluster_id']] = candidate
                used_entities.add(candidate['entity_id'])
        for cluster_id, indices in clusters.items():
            match = accepted.get(cluster_id)
            entity_id = match['entity_id'] if match else f'E{len(entities)+1}'
            if entity_id not in entities:
                entities[entity_id] = dict(clusters=[], members=[])
            entities[entity_id]['clusters'].append(cluster_id)
            for char_index in indices:
                character_id = f'PG{page_index+1}:C{char_index+1}'
                entities[entity_id]['members'].append(dict(page_index=page_index, character_index=char_index,
                                                         character_id=character_id, cluster_id=cluster_id))
                char_to_entity[character_id] = entity_id
            decisions.append(dict(page_index=page_index, cluster_id=cluster_id, entity_id=entity_id,
                                  reason='matched' if match else 'new_entity',
                                  score=match['score'] if match else None, best_pair=match['best_pair'] if match else None))
    return dict(entities=entities, char_to_entity=char_to_entity, decisions=decisions)


In [13]:
entity_results = cached_stage('entities',
    dict(characters=stage_ref('character_cleanup'), threshold=ENTITY_THRESHOLD,
         algorithm='sequential_best_member_one_cluster_per_entity_per_page'),
    lambda: clusterize(character_results, character_embeddings_per_page, ENTITY_THRESHOLD))
identified_char = entity_results['entities']
char_to_entity = entity_results['char_to_entity']
assert len(char_to_entity) == sum(len(r['characters']) for r in character_results)
print('Entities:', len(identified_char))
display(pd.DataFrame(entity_results['decisions']))


RUN   entities
Entities: 10


,page_index,cluster_id,entity_id,reason,score,best_pair
0,0,PG1:CL1,E1,new_entity,NaN,None
1,0,PG1:CL2,E2,new_entity,NaN,None
2,0,PG1:CL3,E3,new_entity,NaN,None
3,0,PG1:CL4,E4,new_entity,NaN,None
4,0,PG1:CL5,E5,new_entity,NaN,None
5,0,PG1:CL6,E6,new_entity,NaN,None
6,0,PG1:CL7,E7,new_entity,NaN,None
7,0,PG1:CL8,E8,new_entity,NaN,None
8,1,PG2:CL1,E7,matched,0.852400,"{'current': 'PG2:C1', 'existing': 'PG1:C9'}"
9,1,PG2:CL2,E2,matched,0.866527,"{'current': 'PG2:C2', 'existing': 'PG1:C2'}"


## 6. Link original text indices to entities

<details>
<summary>How this stage works</summary>

Follow `text_index → character_index → E ID`. Keep one final record per original text box, including empty OCR and unassociated text.

A single associated entity populates `entity_id`. No association gives `None`. Multiple associated entities are preserved with an `ambiguous` status instead of silently choosing one. E IDs are scoped by `sequence_id`.

GT labels remain separate. We do not assume GT dialogue order matches detection order or that E1 equals character1.

</details>

In [14]:
def map_text_entities(results, selections, entity_map, sequence_id, page_names):
    assert len(results) == len(selections) == len(page_names)
    records = []
    for page_index, (result, decisions) in enumerate(zip(results, selections)):
        assert len(result['texts']) == len(result['ocr']) == len(decisions)
        links_by_text = {}
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']) and 0 <= char_index < len(result['characters'])
            links_by_text.setdefault(text_index, []).append(char_index)
        for text_index, (box, text, decision) in enumerate(zip(result['texts'], result['ocr'], decisions)):
            assert decision['page_index'] == page_index and decision['text_index'] == text_index and decision['box'] == box
            assert decision['selected_text'] == text
            links = []
            for char_index in dict.fromkeys(links_by_text.get(text_index, [])):
                char_id = f'PG{page_index+1}:C{char_index+1}'
                links.append(dict(character_index=char_index, character_id=char_id, entity_id=entity_map[char_id]))
            entity_ids = list(dict.fromkeys(link['entity_id'] for link in links))
            records.append(dict(sequence_id=sequence_id, page_index=page_index, page_file=page_names[page_index],
                                text_index=text_index, text_id=f'T{text_index+1}', box=box, text=text,
                                entity_id=entity_ids[0] if len(entity_ids) == 1 else None,
                                entity_ids=entity_ids, character_links=links,
                                association_status='unassociated' if not links else 'linked' if len(entity_ids) == 1 else 'ambiguous',
                                ocr_source=decision['selected_source'], ocr_reason=decision['reason'],
                                needs_review=decision['needs_review'] or len(entity_ids) > 1,
                                disagreement=decision['disagreement']))
    return records


In [15]:
text_entities = cached_stage('text_entities',
    dict(magi=stage_ref('magi_raw'), characters=stage_ref('character_cleanup'),
         selector=stage_ref('ocr_selection'), entities=stage_ref('entities')),
    lambda: map_text_entities(per_page_results, ocr_selection, char_to_entity, SEQUENCE_ID,
                             [path.name for path in PAGE_FILES]))
assert len(text_entities) == sum(len(r['texts']) for r in magi_raw_results)
columns = ['page_index', 'text_id', 'text', 'entity_id', 'association_status',
           'ocr_source', 'needs_review', 'disagreement']
display(pd.DataFrame(text_entities, columns=columns))


RUN   text_entities


,page_index,text_id,text,entity_id,association_status,ocr_source,needs_review,disagreement
0,0,T1,THIS IS OUR OWN THING WE GOT HERE! IT'S GONNA ...,E1,linked,paddle,False,True
1,0,T2,"WHEN WE DO BUSINESS WITH SOMEONE LIKE DINO, IT...",E1,linked,paddle,False,True
2,0,T3,HE CAN'T BE TRUSTED EXCEPT TO TRY AND PUT A CO...,NaN,unassociated,paddle,False,True
3,0,T4,BUT IF YOU WAN TO ROLL OVER FOR HIM...,E1,linked,paddle,False,True
4,0,T5,...AT LEAST SHOW ME ENOUGH RESPECT NOT TO BEG ...,E1,linked,paddle,False,False
5,0,T6,"ISN'T THAT RIGHT, ARTHUR?",E1,linked,paddle,False,False
6,0,T7,e0A,E6,linked,paddle,False,False
7,1,T1,"HEY, YOU'RE THE BOSS.",E6,linked,paddle,False,False
8,1,T2,I'VE GOT NO COMPLAINTS,E6,linked,paddle,False,True
9,1,T3,THAT KEEPS THINGS NICE AND SIMPLE FOR ME,E1,linked,paddle,False,True


## 7. Inspect pages visually

<details>
<summary>How this stage works</summary>

Based on `magiv2_annotated_outputs.ipynb`: panels, text boxes, character boxes, tails, same-character links, text→character links, text→tail links, OCR, and IDs.

Controls add selected/Paddle/Magi OCR views and sequence entity colors/labels. Page selection is one-based in the UI; stored indices remain zero-based. `show_page(...)` works without widgets and can save annotations.

`inspect_text(page_index, text_index)` shows the crop, both full OCRs, validity results, decision reason, and associated entities.

</details>

In [16]:
def center(box):
    x1, y1, x2, y2 = box
    return ((x1+x2)/2, (y1+y2)/2)


def draw_box(ax, box, color, label=None):
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                   edgecolor=color, facecolor='none'))
    if label:
        ax.text(x1, max(0, y1-4), label, fontsize=8, color='white',
                bbox=dict(facecolor='black', alpha=0.7, edgecolor='none'))


def show_page(page_index=0, panels=False, texts=True, characters=True, tails=False,
              same_character=False, text_character=True, text_tail=False,
              ocr=True, ids=True, entities=True, ocr_source='selected', save_path=None):
    result = per_page_results[page_index]
    assert ocr_source in {'selected', 'paddle', 'magi'}
    cmap = plt.get_cmap('tab20', max(len(identified_char), 1))
    colors = {entity_id: cmap(i) for i, entity_id in enumerate(identified_char)}
    page_records = [r for r in text_entities if r['page_index'] == page_index]
    fig, ax = plt.subplots(figsize=(12, 18))
    ax.imshow(chapter_pages[page_index])
    if panels:
        for i, box in enumerate(result.get('panels', [])):
            draw_box(ax, box, 'green', f'PN{i+1}' if ids else None)
    if texts:
        for i, box in enumerate(result['texts']):
            record = page_records[i]
            essential = result.get('is_essential_text', [])
            color = 'red' if i >= len(essential) or essential[i] else 'gray'
            label = f'T{i+1}' if ids else ''
            if entities:
                label += ' | ' + ('/'.join(record['entity_ids']) or 'unassociated')
            if ocr:
                text = (result['ocr'][i] if ocr_source == 'selected' else
                        ocr_selection[page_index][i][f'{ocr_source}_text']) or ''
                label += ' | ' + (text[:62] + '…' if len(text) > 65 else text)
            if record['needs_review']:
                label += ' [review]'
            draw_box(ax, box, color, label.strip(' |'))
    if characters:
        for i, box in enumerate(result['characters']):
            char_id = f'PG{page_index+1}:C{i+1}'
            entity_id = char_to_entity[char_id]
            label = f'C{i+1} / CL{int(result["character_cluster_labels"][i])+1}' if ids else ''
            if entities:
                label += f' | {entity_id}'
            draw_box(ax, box, colors[entity_id] if entities else 'blue', label.strip(' |'))
    if tails:
        for i, box in enumerate(result.get('tails', [])):
            draw_box(ax, box, 'purple', f'TAIL{i+1}' if ids else None)
    if same_character:
        for indices in page_clusters(result, page_index).values():
            for index in indices[1:]:
                a, b = center(result['characters'][indices[0]]), center(result['characters'][index])
                ax.plot([a[0], b[0]], [a[1], b[1]], color='blue', linewidth=1.5)
    for enabled, key, target, style in [
        (text_character, 'text_character_associations', 'characters', '--'),
        (text_tail, 'text_tail_associations', 'tails', ':')]:
        if enabled:
            for text_index, target_index in result.get(key, []):
                if not (0 <= text_index < len(result['texts']) and 0 <= target_index < len(result.get(target, []))):
                    continue
                a, b = center(result['texts'][text_index]), center(result[target][target_index])
                color = 'purple' if target == 'tails' else 'blue'
                if entities and target == 'characters':
                    color = colors[char_to_entity[f'PG{page_index+1}:C{target_index+1}']]
                ax.plot([a[0], b[0]], [a[1], b[1]], color=color, linestyle=style, linewidth=1.5)
                if ids:
                    target_id = f'C{target_index+1}' if target == 'characters' else f'TAIL{target_index+1}'
                    ax.text((a[0]+b[0])/2, (a[1]+b[1])/2, f'T{text_index+1}→{target_id}', fontsize=7)
    ax.set_title(f'{SEQUENCE_ID} | PG{page_index+1}: {PAGE_FILES[page_index].name} | {ocr_source} OCR')
    ax.axis('off')
    fig.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)


def inspect_text(page_index=0, text_index=0):
    decision = ocr_selection[page_index][text_index]
    crop, _ = crop_box(chapter_pages[page_index], decision['box'])
    if crop is not None:
        display(Image.fromarray(crop))
    record = next(r for r in text_entities if r['page_index'] == page_index and r['text_index'] == text_index)
    print(json.dumps(dict(ocr_decision=decision, text_entity=record), ensure_ascii=False, indent=2))


In [17]:
page_control = widgets.Dropdown(options=[(f'PG{i+1} — {p.name}', i) for i, p in enumerate(PAGE_FILES)], description='Page:')
source_control = widgets.Dropdown(options=['selected', 'paddle', 'magi'], description='OCR:')
flags = {name: widgets.Checkbox(value=value, description=name.replace('_', ' '), indent=False)
         for name, value in dict(panels=False, texts=True, characters=True, tails=False,
                                 same_character=False, text_character=True, text_tail=False,
                                 ocr=True, ids=True, entities=True).items()}
controls = dict(page_index=page_control, ocr_source=source_control, **flags)
viewer_output = widgets.interactive_output(show_page, controls)
display(widgets.VBox([widgets.HBox([page_control, source_control]),
                      widgets.HBox(list(flags.values())[:5]), widgets.HBox(list(flags.values())[5:])]), viewer_output)


Output()

In [18]:
# Direct calls, if widgets are unavailable:
# show_page(0, panels=True, tails=True, same_character=True, text_tail=True)
# show_page(0, ocr_source='magi')
# show_page(0, save_path=CACHE_DIR / 'annotations' / '01.png')
# inspect_text(page_index=0, text_index=0)
# for page_index in range(len(chapter_pages)):
#     show_page(page_index)


## 8. Ground truth and cache inspection

<details>
<summary>How this stage works</summary>

GT labels are independent supervision, not inferred entity names. No automatic GT matching or text-order alignment is performed.

The table shows cached files, timestamps, and signatures. For another sequence, change `SEQUENCE_ID` and rerun. Its cache is isolated. Later, `text_entities.json` records can be combined into a training export without rewriting inference caches.

</details>

In [19]:
gt_rows = [dict(page_index=page_index, gt_index=gt_index, speaker=label['speaker'], text=label['text'])
           for page_index, labels in enumerate(sequence_labels['pages']) for gt_index, label in enumerate(labels)]
print('Original ground truth labels:')
display(pd.DataFrame(gt_rows, columns=['page_index', 'gt_index', 'speaker', 'text']))
print('Cache stages:')
display(pd.DataFrame([dict(stage=name, file=str(CACHE_DIR / entry['file']),
                          created_at=entry['created_at'], signature=entry['signature'][:16])
                      for name, entry in manifest['stages'].items()]))
print('Final text records:', len(text_entities), '| entities:', len(identified_char))


Original ground truth labels:


,page_index,gt_index,speaker,text
0,0,0,character1,THIS IS OUR OWN THING WE GOT HERE! IT'S GONNA ...
1,0,1,character1,"WHEN WE DO BUSINESS WITH SOMEONE LIKE DINO, IT..."
2,0,2,character1,HE CAN'T BE TRUSTED -- EXCEPT TO TRY AND PUT A...
3,0,3,character1,BUT IF YOU WANT TO ROLL OVER FOR HIM...
4,0,4,character1,...AT LEAST SHOW ME ENOUGH RESPECT NOT TO BEG ...
5,0,5,character1,"ISN'T THAT RIGHT, ARTHUR?"
6,0,6,character2,...
7,1,0,character2,"HEY, YOU'RE THE BOSS."
8,1,1,character2,I'VE GOT NO COMPLAINTS.
9,1,2,character1,THAT KEEPS THINGS NICE AND SIMPLE FOR ME.


Cache stages:


,stage,file,created_at,signature
0,magi_raw,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-06T04:41:36.813937+00:00,6dbced6433738f7b
1,embeddings,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-06T04:41:37.288131+00:00,a4baa186c2a145b7
2,paddle_ocr,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-06T04:41:47.438549+00:00,f98064567c58e35b
3,ocr_selection,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-06T04:41:47.493046+00:00,87375aeee1642178
4,character_cleanup,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-06T04:41:47.669397+00:00,0adc711447347c52
5,entities,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-06T04:41:47.768376+00:00,47e8aec8a8092e95
6,text_entities,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-06T04:41:47.822893+00:00,a5ae126e68b27248


Final text records: 28 | entities: 10


## 9. Match selected OCR to ground truth

<details>
<summary>Matching stages, reading order, and repeated GT matches</summary>

This is for inspecting OCR performance. **Magi's original text-box order is the reading order.** No extra reading-order model, no box reordering, and no changes to existing OCR or entity assignments.

1. **Direct:** Compare one normalized OCR bubble to all GT lines on its page using full-string fuzzy similarity. Accept the best at **95+**.
2. **Adjacent join:** For unresolved bubbles, try consecutive pairs `T_i + T_(i+1)` on the same page. Both must still be unresolved. Accept full-string similarity at **95+**. Stronger overlapping pairs win; each bubble belongs to one selected match group.
3. **Fallback:** For remaining single bubbles, combine partial ratio, token overlap coefficient, and character n-gram overlap. Default weights are **40% / 30% / 30%**, acceptance **75+**. All fallback matches need review. Short fragments with fewer than three normalized letters/digits remain unmatched unless a strict stage already accepted them.

**GT reuse is allowed:** two independently matching OCR bubbles may both map to the same GT line. This is many-to-one alignment, not one-to-one label reconstruction. Joined groups retain both original bubble indices and entity/association information.

Normalization uses Unicode NFKC, case folding, punctuation as separators in ordinary dialogue, and collapsed whitespace. Punctuation/symbol-only dialogue is preserved, so identical punctuation in OCR and GT can match directly. No word-specific corrections. Full similarity uses Python's `SequenceMatcher` ratio (0-100). Partial ratio is the highest ratio against a same-length contiguous window of the longer text. Overlap scores use set intersection divided by the smaller set size; n-grams include word-boundary spaces.

`gt_matches.json` stores match groups, one record per original bubble, GT coverage, duplicate GT matches, scores, and matching stages. An unmatched record retains its best fallback candidate when available. Match/coverage rates are diagnostics, not certified OCR or speaker accuracy.

</details>

In [20]:
from difflib import SequenceMatcher

GT_MATCH_SETTINGS = dict(
    strict_threshold=95.0,
    fallback_threshold=75.0,
    fallback_weights=dict(partial_ratio=0.40, token_overlap=0.30, ngram_overlap=0.30),
    ngram_size=3,
    fallback_min_chars=3,
    allow_gt_reuse=True,
    scoring='difflib_full_and_window_partial_v1',
    normalization='nfkc_casefold_alnum_preserve_symbol_only_v2',
)
FORCE.setdefault('gt_matches', False)  # Set True to recompute just this stage.
STAGE_VERSIONS['gt_matches'] = 2


In [21]:
def normalize_gt_text(text):
    text = unicodedata.normalize('NFKC', text or '').casefold()
    words = ' '.join(''.join(char if char.isalnum() else ' ' for char in text).split())
    if words:
        return words
    # Punctuation/symbol-only dialogue is meaningful, not empty OCR.
    symbols = ''.join(char for char in text
                      if char.isspace() or unicodedata.category(char)[0] in {'P', 'S'})
    return ' '.join(symbols.split())


def full_fuzzy_score(a, b):
    if not a or not b:
        return 0.0
    return 100.0 * SequenceMatcher(None, a, b, autojunk=False).ratio()


def partial_fuzzy_score(a, b):
    if not a or not b:
        return 0.0
    short, long = (a, b) if len(a) <= len(b) else (b, a)
    if short in long:
        return 100.0
    return max(full_fuzzy_score(short, long[start:start+len(short)])
               for start in range(len(long)-len(short)+1))


def overlap_score(a, b):
    if not a or not b:
        return 0.0
    return 100.0 * len(a & b) / min(len(a), len(b))


def text_ngrams(text, size):
    return {text[i:i+size] for i in range(max(0, len(text)-size+1))}


def gt_scores(ocr_text, gt_text, settings):
    a, b = normalize_gt_text(ocr_text), normalize_gt_text(gt_text)
    scores = dict(full_ratio=full_fuzzy_score(a, b), partial_ratio=partial_fuzzy_score(a, b),
                  token_overlap=overlap_score(set(a.split()), set(b.split())),
                  ngram_overlap=overlap_score(text_ngrams(a, settings['ngram_size']),
                                              text_ngrams(b, settings['ngram_size'])))
    scores['fallback_score'] = sum(settings['fallback_weights'][key] * scores[key]
                                   for key in settings['fallback_weights'])
    return scores


def best_gt_candidate(text, gt_lines, settings, score_key):
    best = None
    for gt_index, gt in enumerate(gt_lines):
        if not normalize_gt_text(gt['text']):
            continue
        scores = gt_scores(text, gt['text'], settings)
        if best is None or scores[score_key] > best['scores'][score_key]:
            best = dict(gt_index=gt_index, gt_text=gt['text'], gt_speaker=gt['speaker'], scores=scores)
    return best


In [22]:
def match_ocr_to_gt(results, labels, text_records, settings):
    assert len(results) == len(labels['pages']), 'GT/page count mismatch'
    assert settings['allow_gt_reuse'], 'This performance stage allows repeated GT matches'
    assert 0 <= settings['strict_threshold'] <= 100 and 0 <= settings['fallback_threshold'] <= 100
    assert settings['ngram_size'] >= 1 and settings['fallback_min_chars'] >= 0
    weights = settings['fallback_weights']
    assert set(weights) == {'partial_ratio', 'token_overlap', 'ngram_overlap'}
    assert all(weight >= 0 for weight in weights.values()) and abs(sum(weights.values())-1) < 1e-9
    metadata = {(r['page_index'], r['text_index']): r for r in text_records}
    expected = {(page_index, text_index) for page_index, result in enumerate(results)
                for text_index in range(len(result['texts']))}
    assert set(metadata) == expected and len(metadata) == len(text_records), 'Text metadata alignment mismatch'
    groups, bubble_matches, gt_coverage = [], [], []

    for page_index, result in enumerate(results):
        texts, gt_lines = result['ocr'], labels['pages'][page_index]
        assert len(texts) == len(result['texts']), 'OCR/box count mismatch'
        assignments = {}

        def add_group(indices, stage, candidate, reason):
            text = ' '.join(texts[index] for index in indices)
            accepted = stage != 'unmatched'
            group = dict(sequence_id=labels['sequence_id'], page_index=page_index,
                         text_indices=list(indices), bubble_ids=[f'PG{page_index+1}:T{i+1}' for i in indices],
                         ocr_text=text, normalized_ocr=normalize_gt_text(text), stage=stage, reason=reason,
                         gt_index=candidate['gt_index'] if accepted else None,
                         gt_text=candidate['gt_text'] if accepted else None,
                         gt_speaker=candidate['gt_speaker'] if accepted else None,
                         scores=candidate['scores'] if candidate else None,
                         best_candidate=candidate, needs_review=stage in {'fallback', 'unmatched'})
            for index in indices:
                assert index not in assignments, 'Bubble was matched twice'
                assignments[index] = group
            return group

        # Stage 1: each bubble gets its own best match. GT lines are never reserved.
        for text_index, text in enumerate(texts):
            if normalize_gt_text(text):
                candidate = best_gt_candidate(text, gt_lines, settings, 'full_ratio')
                if candidate and candidate['scores']['full_ratio'] >= settings['strict_threshold']:
                    add_group([text_index], 'direct', candidate, 'strict_single')

        # Stage 2: only genuinely consecutive unresolved pairs, never across pages.
        joined_candidates = []
        for text_index in range(len(texts)-1):
            indices = [text_index, text_index+1]
            if any(index in assignments for index in indices):
                continue
            if not all(normalize_gt_text(texts[index]) for index in indices):
                continue
            joined_text = ' '.join(texts[index] for index in indices)
            candidate = best_gt_candidate(joined_text, gt_lines, settings, 'full_ratio')
            if candidate and candidate['scores']['full_ratio'] >= settings['strict_threshold']:
                joined_candidates.append((indices, candidate))
        joined_candidates.sort(key=lambda item: item[1]['scores']['full_ratio'], reverse=True)
        for indices, candidate in joined_candidates:
            if not any(index in assignments for index in indices):
                add_group(indices, 'adjacent_join', candidate, 'strict_adjacent_pair')

        # Stage 3: single remaining fragments; tentative matches always need review.
        for text_index, text in enumerate(texts):
            if text_index in assignments:
                continue
            normalized = normalize_gt_text(text)
            candidate = best_gt_candidate(text, gt_lines, settings, 'fallback_score') if normalized else None
            signal = sum(char.isalnum() for char in normalized)
            if candidate and signal >= settings['fallback_min_chars'] and candidate['scores']['fallback_score'] >= settings['fallback_threshold']:
                add_group([text_index], 'fallback', candidate, 'partial_overlap_fragment')
            else:
                reason = ('empty_ocr' if not normalized else 'no_gt_lines' if not candidate else
                          'fragment_too_short' if signal < settings['fallback_min_chars'] else 'below_fallback_threshold')
                add_group([text_index], 'unmatched', candidate, reason)

        # Store groups and bubble rows in ORIGINAL order, irrespective of matching priority.
        for text_index in range(len(texts)):
            group = assignments[text_index]
            if text_index == group['text_indices'][0]:
                group['match_id'] = f'M{len(groups)+1}'
                group['entity_ids'] = list(dict.fromkeys(entity for index in group['text_indices']
                    for entity in metadata[(page_index, index)]['entity_ids']))
                groups.append(group)
            original = metadata[(page_index, text_index)]
            assert original['text'] == texts[text_index] and original['box'] == result['texts'][text_index]
            bubble_matches.append(dict(sequence_id=labels['sequence_id'], page_index=page_index,
                text_index=text_index, text_id=original['text_id'], box=original['box'], ocr_text=texts[text_index],
                match_id=group['match_id'], group_text_indices=group['text_indices'], stage=group['stage'],
                gt_index=group['gt_index'], gt_text=group['gt_text'], gt_speaker=group['gt_speaker'],
                scores=group['scores'], reason=group['reason'], needs_review=group['needs_review'],
                entity_id=original['entity_id'], entity_ids=original['entity_ids'],
                character_links=original['character_links'], association_status=original['association_status'],
                ocr_source=original['ocr_source'], ocr_needs_review=original['needs_review']))
        for gt_index, gt in enumerate(gt_lines):
            matched_groups = [g for g in groups if g['page_index'] == page_index and g['gt_index'] == gt_index]
            gt_coverage.append(dict(page_index=page_index, gt_index=gt_index, gt_text=gt['text'],
                gt_speaker=gt['speaker'], match_ids=[g['match_id'] for g in matched_groups],
                group_count=len(matched_groups), bubble_count=sum(len(g['text_indices']) for g in matched_groups),
                strict_group_count=sum(g['stage'] in {'direct', 'adjacent_join'} for g in matched_groups)))
    return dict(settings=settings, groups=groups, bubble_matches=bubble_matches, gt_coverage=gt_coverage)


In [23]:
gt_matching = cached_stage('gt_matches',
    dict(text_entities=stage_ref('text_entities'), selector=stage_ref('ocr_selection'),
         labels_sha256=digest(sequence_labels), matching=GT_MATCH_SETTINGS,
         python=sys.version.split()[0]),
    lambda: match_ocr_to_gt(per_page_results, sequence_labels, text_entities, GT_MATCH_SETTINGS))
gt_match_groups = gt_matching['groups']
gt_bubble_matches = gt_matching['bubble_matches']
assert len(gt_bubble_matches) == sum(len(result['texts']) for result in per_page_results)
print('Cached:', CACHE_DIR / 'gt_matches.json')


RUN   gt_matches
Cached: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache\seq_0dc2e3d874c804fc\gt_matches.json


## 10. Inspect OCR-to-GT matching performance

<details>
<summary>What these tables measure</summary>

The stage table counts original bubbles (joined groups contribute two bubbles). GT coverage counts unique GT lines, so repeated OCR matches do not inflate coverage. Strict coverage excludes tentative fallback matches. The repeated-match table shows GT lines accepted by more than one group.

These are matching/coverage diagnostics, not ground-truth OCR accuracy: normalization ignores punctuation/case, and fuzzy or partial alignment can accept incorrect text. Entity IDs and GT speakers are displayed alongside each other but no speaker-identity accuracy is inferred.

</details>

In [24]:
stage_order = ['direct', 'adjacent_join', 'fallback', 'unmatched']
stage_counts = {stage: sum(r['stage'] == stage for r in gt_bubble_matches) for stage in stage_order}
total_bubbles = len(gt_bubble_matches)
summary_rows = [dict(stage=stage, bubbles=stage_counts[stage],
                     percent=round(100*stage_counts[stage]/total_bubbles, 2) if total_bubbles else 0)
                for stage in stage_order]
display(pd.DataFrame(summary_rows))
coverage = gt_matching['gt_coverage']
print('OCR bubbles:', total_bubbles, '| dialogue match groups:', len(gt_match_groups))
print('GT lines:', len(coverage), '| strict covered:', sum(r['strict_group_count'] > 0 for r in coverage),
      '| covered including fallback:', sum(r['group_count'] > 0 for r in coverage))
print('GT lines with multiple independently matched groups:', sum(r['group_count'] > 1 for r in coverage))
display(pd.DataFrame([r for r in coverage if r['group_count'] > 1],
                     columns=['page_index', 'gt_index', 'gt_text', 'gt_speaker', 'group_count', 'bubble_count', 'match_ids']))


,stage,bubbles,percent
0,direct,25,89.29
1,adjacent_join,0,0.00
2,fallback,0,0.00
3,unmatched,3,10.71


OCR bubbles: 28 | dialogue match groups: 28
GT lines: 27 | strict covered: 25 | covered including fallback: 25
GT lines with multiple independently matched groups: 0


,page_index,gt_index,gt_text,gt_speaker,group_count,bubble_count,match_ids


In [25]:
def show_gt_matches(stage='all'):
    rows = []
    for group in gt_match_groups:
        if stage != 'all' and group['stage'] != stage:
            continue
        scores = group['scores'] or {}
        rows.append(dict(match_id=group['match_id'], page=group['page_index']+1,
                         bubbles=', '.join(group['bubble_ids']), stage=group['stage'],
                         ocr=group['ocr_text'], gt_index=group['gt_index'], gt=group['gt_text'],
                         speaker=group['gt_speaker'], entities=', '.join(group['entity_ids']),
                         full_ratio=round(scores.get('full_ratio', 0), 2),
                         partial_ratio=round(scores.get('partial_ratio', 0), 2),
                         token_overlap=round(scores.get('token_overlap', 0), 2),
                         ngram_overlap=round(scores.get('ngram_overlap', 0), 2),
                         fallback_score=round(scores.get('fallback_score', 0), 2),
                         review=group['needs_review'], reason=group['reason']))
    display(pd.DataFrame(rows))


def inspect_gt_match(match_id='M1'):
    group = next((g for g in gt_match_groups if g['match_id'] == match_id), None)
    if group is None:
        print(f'No match group {match_id}')
        return
    for text_index in group['text_indices']:
        print(f"PG{group['page_index']+1}:T{text_index+1}")
        crop, _ = crop_box(chapter_pages[group['page_index']], per_page_results[group['page_index']]['texts'][text_index])
        if crop is not None:
            display(Image.fromarray(crop))
    print(json.dumps(group, ensure_ascii=False, indent=2))
    display(pd.DataFrame([r for r in gt_bubble_matches if r['match_id'] == match_id]))


In [26]:
gt_stage_control = widgets.Dropdown(options=['all', 'direct', 'adjacent_join', 'fallback', 'unmatched'],
                                   description='Match stage:')
gt_match_output = widgets.interactive_output(show_gt_matches, dict(stage=gt_stage_control))
display(gt_stage_control, gt_match_output)

# Full scores, GT speaker, entity associations, and both crops for joined groups:
# inspect_gt_match('M1')
# show_gt_matches('unmatched')  # Direct call if widgets are unavailable.


Dropdown(description='Match stage:', options=('all', 'direct', 'adjacent_join', 'fallback', 'unmatched'), valu…

Output()

## 11. Final page dialogue JSON

<details>
<summary>Essential text only, original order, and output files</summary>

For each page, emit an array of `{"speaker": "E1", "text": "GO."}` objects. **Text is selected OCR; speaker is the associated entity, not the GT speaker.** Unassociated or ambiguous entities use `null`.

Filter out only bubbles whose Magi `is_essential_text` flag is false. Walk the original text indices from first to last; do not sort, merge, or renumber them. Essential bubbles with empty OCR stay included. Page order and empty page arrays are preserved. Joined GT matches do not merge bubbles in this export.

Before exporting, check that text boxes, association indices, and text-entity record order still match the original Magi results. Missing or misaligned essential flags stop the export rather than guessing. No original page results or cached inference data are modified.

**Output:** `cache/<sequence_id>/dialogue_pages.json` is the plain array of page arrays, with exactly `speaker` and `text` per object.

**Cache/audit:** `dialogue_export.json` contains the stage signature and export data, including retained and removed original text indices. The export depends on Magi and text-entity outputs; it does not depend on GT matching. Indices in the audit are zero-based.

</details>


In [27]:
def export_dialogue_pages(results, raw_results, text_records, character_pages):
    assert len(results) == len(raw_results), 'Original page count changed'
    expected_order = [(page_index, text_index)
                      for page_index, result in enumerate(raw_results)
                      for text_index in range(len(result['texts']))]
    actual_order = [(r['page_index'], r['text_index']) for r in text_records]
    assert actual_order == expected_order, 'Original text-entity reading order changed'
    metadata = {(r['page_index'], r['text_index']): r for r in text_records}
    pages, kept_indices, removed_indices = [], [], []

    for page_index, (result, raw) in enumerate(zip(results, raw_results)):
        assert result['texts'] == raw['texts'], f'PG{page_index+1}: original text boxes/order changed'
        assert result['characters'] == character_pages[page_index]['characters'], 'Cleaned character boxes changed'
        assert result['text_character_associations'] == character_pages[page_index]['text_character_associations'], 'Cleaned associations changed'
        assert len(result['ocr']) == len(raw['texts']), 'OCR/text count mismatch'
        essential = raw.get('is_essential_text')
        assert essential is not None and len(essential) == len(raw['texts']), 'Missing or misaligned essential flags'
        assert all(isinstance(flag, bool) for flag in essential), 'Expected boolean essential flags'
        page, kept, removed = [], [], []
        for text_index, text in enumerate(result['ocr']):
            record = metadata[(page_index, text_index)]
            assert record['box'] == raw['texts'][text_index] and record['text'] == text, 'Text-entity alignment changed'
            assert isinstance(text, str), 'Selected OCR must be a string'
            if not essential[text_index]:
                removed.append(text_index)
                continue
            speaker = record['entity_id']
            assert speaker is None or isinstance(speaker, str), 'Entity ID must be a string or null'
            page.append(dict(speaker=speaker, text=text))
            kept.append(text_index)
        assert kept == [index for index, flag in enumerate(essential) if flag], 'Retained reading order changed'
        pages.append(page)
        kept_indices.append(kept)
        removed_indices.append(removed)
    return dict(pages=pages, kept_text_indices=kept_indices, removed_text_indices=removed_indices)


In [28]:
FORCE.setdefault('dialogue_export', False)
STAGE_VERSIONS.setdefault('dialogue_export', 1)

# Check current in-memory alignment even when the export cache exists.
current_dialogue_export = export_dialogue_pages(per_page_results, magi_raw_results, text_entities, character_results)
dialogue_export = cached_stage('dialogue_export',
    dict(magi=stage_ref('magi_raw'), text_entities=stage_ref('text_entities'),
         current_data_sha256=digest(current_dialogue_export),
         filter='magi_is_essential_text', order='original_text_indices', speaker='entity_id_or_null'),
    lambda: current_dialogue_export)
dialogue_pages = dialogue_export['pages']
DIALOGUE_EXPORT_PATH = CACHE_DIR / 'dialogue_pages.json'
write_json(DIALOGUE_EXPORT_PATH, dialogue_pages)
print('Saved:', DIALOGUE_EXPORT_PATH)


RUN   dialogue_export
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache\seq_0dc2e3d874c804fc\dialogue_pages.json


In [29]:
export_summary = [dict(page=page_index+1, page_file=PAGE_FILES[page_index].name,
                       original_bubbles=len(magi_raw_results[page_index]['texts']),
                       exported_bubbles=len(page),
                       retained_text_ids=[f'T{i+1}' for i in dialogue_export['kept_text_indices'][page_index]],
                       removed_text_ids=[f'T{i+1}' for i in dialogue_export['removed_text_indices'][page_index]],
                       null_speakers=sum(item['speaker'] is None for item in page))
                  for page_index, page in enumerate(dialogue_pages)]
display(pd.DataFrame(export_summary))
print(json.dumps(dialogue_pages, ensure_ascii=False, indent=2))


,page,page_file,original_bubbles,exported_bubbles,retained_text_ids,removed_text_ids,null_speakers
0,1,01.png,7,7,"[T1, T2, T3, T4, T5, T6, T7]",[],1
1,2,02.png,7,7,"[T1, T2, T3, T4, T5, T6, T7]",[],0
2,3,03.png,14,13,"[T1, T2, T3, T4, T5, T6, T8, T9, T10, T11, T12...",[T7],0


[
  [
    {
      "speaker": "E1",
      "text": "THIS IS OUR OWN THING WE GOT HERE! IT'S GONNA STAY OUR OWN THING!"
    },
    {
      "speaker": "E1",
      "text": "WHEN WE DO BUSINESS WITH SOMEONE LIKE DINO, IT'S ON EVEN TERMS!"
    },
    {
      "speaker": null,
      "text": "HE CAN'T BE TRUSTED EXCEPT TO TRY AND PUT A COLLAR ON YOU, UNDER- STAND?!"
    },
    {
      "speaker": "E1",
      "text": "BUT IF YOU WAN TO ROLL OVER FOR HIM..."
    },
    {
      "speaker": "E1",
      "text": "...AT LEAST SHOW ME ENOUGH RESPECT NOT TO BEG MY FUCKING PARDON."
    },
    {
      "speaker": "E1",
      "text": "ISN'T THAT RIGHT, ARTHUR?"
    },
    {
      "speaker": "E6",
      "text": "e0A"
    }
  ],
  [
    {
      "speaker": "E6",
      "text": "HEY, YOU'RE THE BOSS."
    },
    {
      "speaker": "E6",
      "text": "I'VE GOT NO COMPLAINTS"
    },
    {
      "speaker": "E1",
      "text": "THAT KEEPS THINGS NICE AND SIMPLE FOR ME"
    },
    {
      "speaker": "E7",
      "text":

In [30]:
print(json.dumps(sequence_labels, ensure_ascii=False, indent=2))


{
  "sequence_id": "seq_0dc2e3d874c804fc",
  "pages": [
    [
      {
        "speaker": "character1",
        "text": "THIS IS OUR OWN THING WE GOT HERE! IT'S GONNA STAY OUR OWN THING!"
      },
      {
        "speaker": "character1",
        "text": "WHEN WE DO BUSINESS WITH SOMEONE LIKE DINO, IT'S ON EVEN TERMS!"
      },
      {
        "speaker": "character1",
        "text": "HE CAN'T BE TRUSTED -- EXCEPT TO TRY AND PUT A COLLAR ON YOU, UNDERSTAND?!"
      },
      {
        "speaker": "character1",
        "text": "BUT IF YOU WANT TO ROLL OVER FOR HIM..."
      },
      {
        "speaker": "character1",
        "text": "...AT LEAST SHOW ME ENOUGH RESPECT NOT TO BEG MY FUCKING PARDON."
      },
      {
        "speaker": "character1",
        "text": "ISN'T THAT RIGHT, ARTHUR?"
      },
      {
        "speaker": "character2",
        "text": "..."
      }
    ],
    [
      {
        "speaker": "character2",
        "text": "HEY, YOU'RE THE BOSS."
      },
      {
        "spe